Input: CSV files of consecutive months of Divvy trip data

Output: Parquet file of cleaned dataset

In [9]:
from pathlib import Path # for cleaner imports of data

RAW_DIR = Path("../data/raw") # care about reading from the raw data directory

csv_paths = sorted(RAW_DIR.glob("*-divvy-tripdata.csv"))  # want to ensure each month file is in increasing order

print(f"Found {len(csv_paths)} from {RAW_DIR.resolve()}")  # print number of files
for file in csv_paths[:10]:
    print(f"File Name: {file}")  # print the first 10 file names

Found 10 from /Users/rishikumar/divvy_project/data/raw
File Name: ../data/raw/202501-divvy-tripdata.csv
File Name: ../data/raw/202502-divvy-tripdata.csv
File Name: ../data/raw/202503-divvy-tripdata.csv
File Name: ../data/raw/202504-divvy-tripdata.csv
File Name: ../data/raw/202505-divvy-tripdata.csv
File Name: ../data/raw/202506-divvy-tripdata.csv
File Name: ../data/raw/202507-divvy-tripdata.csv
File Name: ../data/raw/202508-divvy-tripdata.csv
File Name: ../data/raw/202509-divvy-tripdata.csv
File Name: ../data/raw/202510-divvy-tripdata.csv


In [10]:
import pandas as pd

# want to ensure consistent columns across all files

# get the columns of the first month
base_cols = list(pd.read_csv(csv_paths[0], nrows=0).columns) 

# loop through the remaining months one by one, retrieve columns and check if matches

all_match = True # for matching or not

for file in csv_paths[1:]:
    cols = list(pd.read_csv(file, nrows=0).columns)
    if cols != base_cols:
        print(f"Mismatch found in {file.name}")
        print(f"Columns: {cols}")
        all_match = False

if all_match == True:
    print("All files have identical columns")
    for c in base_cols:
        print(c)
    sample_row = pd.read_csv(csv_paths[0], nrows=1).iloc[0] # sample row as subset first month df to first row 
    print(f"\n{sample_row}")


All files have identical columns
ride_id
rideable_type
started_at
ended_at
start_station_name
start_station_id
end_station_name
end_station_id
start_lat
start_lng
end_lat
end_lng
member_casual

ride_id                        7569BC890583FCD7
rideable_type                      classic_bike
started_at              2025-01-21 17:23:54.538
ended_at                2025-01-21 17:37:52.015
start_station_name    Wacker Dr & Washington St
start_station_id                   KA1503000072
end_station_name           McClurg Ct & Ohio St
end_station_id                     TA1306000029
start_lat                             41.883143
start_lng                            -87.637242
end_lat                               41.892592
end_lng                              -87.617289
member_casual                            member
Name: 0, dtype: object


Allow notebook to import modules from the src directory.

Because this notebook runs from the notebooks folder, we add the project root
to Python’s module search path so `src.data_cleaning` can be imported.

In [17]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

Load each monthly CSV and apply the standardized cleaning function.

In [18]:
from src.data_cleaning import clean_divvy_dataframe

cleaned_dfs = []

for file in csv_paths:
    df = pd.read_csv(file)
    df = clean_divvy_dataframe(df)
    cleaned_dfs.append(df)

print(f"Cleaned {len(cleaned_dfs)} monthly files")

Cleaned 10 monthly files


In [19]:
df_all = pd.concat(cleaned_dfs, ignore_index=True)

print(df_all.shape)
df_all.head()

(3362086, 13)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member
1,013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member
2,EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member
3,EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member
4,7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member


In [20]:
OUTPUT_PATH = Path("../data/cleaned/divvy_clean.parquet")

df_all.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved cleaned parquet to {OUTPUT_PATH.resolve()}")

Saved cleaned parquet to /Users/rishikumar/divvy_project/data/cleaned/divvy_clean.parquet
